# PathTrie behavior tests

This notebook exercises `dotvectordb.data_structures.trie.PathTrie` without changing repository code. Run **Kernel → Restart Kernel and Run All Cells** after editing the implementation.

The expected contract used here is: paths are sequences of relationship IDs; `add` reports whether a path was accepted; exact lookup only matches a terminal path; prefix lookup reports whether the queried sequence is a prefix of a stored path; and when one path extends another, the trie retains the longer path.

In [4]:
# Make imports work whether Jupyter starts here or in a child directory.
from pathlib import Path
import importlib.util

working_dir = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in (working_dir, *working_dir.parents)
     if (candidate / 'dotvectordb' / 'data_structures' / 'trie.py').is_file()),
    None,
)
assert repo_root is not None, 'Could not find the DotVectorDB repository root.'
trie_file = repo_root / 'dotvectordb' / 'data_structures' / 'trie.py'
spec = importlib.util.spec_from_file_location('path_trie_under_test', trie_file)
trie_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(trie_module)
PathTrie = trie_module.PathTrie

print(f'Testing PathTrie from: {trie_file}')

Testing PathTrie from: /home/x/DotVectorDB/dotvectordb/data_structures/trie.py


## Test definitions

Each test creates a fresh trie, so a failure cannot contaminate later results. The lightweight runner records every failure instead of stopping at the first one.

In [7]:
def test_new_trie_has_no_nonempty_path(verbose=False):
    trie = PathTrie()

    if verbose:
        trie.display()

    assert not trie.contains_prefix([1])
    assert not trie.contains_exact([1])
    assert [1] not in trie


def test_add_and_exact_lookup(verbose=False):
    trie = PathTrie()

    assert trie.add([10, 20, 30]) is True

    if verbose:
        trie.display()

    assert trie.contains_exact([10, 20, 30]) is True
    assert trie.contains_exact([10, 20]) is False
    assert trie.contains_exact([10, 20, 99]) is False


def test_prefix_lookup_and_in_operator(verbose=False):
    trie = PathTrie()

    trie.add(['r1', 'r2', 'r3'])

    if verbose:
        trie.display()

    assert trie.contains_prefix(['r1']) is True
    assert trie.contains_prefix(['r1', 'r2']) is True
    assert ['r1', 'r2'] in trie
    assert trie.contains_prefix(['r1', 'other']) is False
    assert ['other'] not in trie


def test_duplicate_is_rejected_without_mutation(verbose=False):
    trie = PathTrie()

    assert trie.add([1, 2, 3]) is True

    if verbose:
        print("\nBEFORE DUPLICATE:")
        trie.display()

    assert trie.add([1, 2, 3]) is False

    if verbose:
        print("\nAFTER DUPLICATE:")
        trie.display()

    assert trie.contains_exact([1, 2, 3]) is True


def test_shorter_candidate_is_rejected_when_longer_path_exists(verbose=False):
    trie = PathTrie()

    trie.add([1, 2, 3])

    if verbose:
        print("\nBEFORE SHORTER CANDIDATE:")
        trie.display()

    assert trie.add([1, 2]) is False

    if verbose:
        print("\nAFTER SHORTER CANDIDATE:")
        trie.display()

    assert trie.contains_exact([1, 2]) is False
    assert trie.contains_exact([1, 2, 3]) is True


def test_longer_candidate_replaces_shorter_terminal_path(verbose=True):
    trie = PathTrie()

    assert trie.add([1, 2]) is True

    if verbose:
        print("\nBEFORE LONGER PATH:")
        trie.display()

    assert trie.add([1, 2, 3]) is True

    if verbose:
        print("\nAFTER LONGER PATH:")
        trie.display()

    assert trie.contains_exact([1, 2]) is False
    assert trie.contains_exact([1, 2, 3]) is True


def test_independent_branches_are_preserved(verbose=False):
    trie = PathTrie()

    paths = [(1, 2, 3), (1, 4), (9, 8)]

    assert all(trie.add(path) is True for path in paths)

    if verbose:
        trie.display()

    assert all(trie.contains_exact(path) is True for path in paths)
    assert trie.contains_exact((1, 2, 4)) is False


def test_accepts_tuple_and_string_relationship_ids(verbose=False):
    trie = PathTrie()

    path = ('rel-alpha', 'rel-beta')

    assert trie.add(path) is True

    if verbose:
        trie.display()

    assert trie.contains_exact(path) is True
    assert trie.contains_prefix(('rel-alpha',)) is True

In [8]:
# Discover and run every test function defined above.
tests = sorted(
    (name, value) for name, value in globals().items()
    if name.startswith('test_') and callable(value)
)
results = []

for name, test in tests:
    try:
        test()
    except Exception as exc:
        results.append((name, False, f'{type(exc).__name__}: {exc}'))
        print(f'FAIL  {name} — {type(exc).__name__}: {exc}')
    else:
        results.append((name, True, ''))
        print(f'PASS  {name}')

passed = sum(passed for _, passed, _ in results)
failed = len(results) - passed
print(f'\nSummary: {passed}/{len(results)} passed; {failed} failed.')

PASS  test_accepts_tuple_and_string_relationship_ids
PASS  test_add_and_exact_lookup
PASS  test_duplicate_is_rejected_without_mutation
PASS  test_independent_branches_are_preserved

BEFORE LONGER PATH:
ROOT
1
  2 [TERMINAL]

AFTER LONGER PATH:
ROOT
1
  2
    3 [TERMINAL]
PASS  test_longer_candidate_replaces_shorter_terminal_path
PASS  test_new_trie_has_no_nonempty_path
PASS  test_prefix_lookup_and_in_operator
PASS  test_shorter_candidate_is_rejected_when_longer_path_exists

Summary: 8/8 passed; 0 failed.


In [4]:
# Final gate: this cell is green only when the complete suite passes.
failures = [f'{name}: {detail}' for name, ok, detail in results if not ok]
assert not failures, 'PathTrie failures:\n- ' + '\n- '.join(failures)
print('All PathTrie tests passed.')

All PathTrie tests passed.
